# K-ABENA — Niveau 1 : NLP Classification (TensorFlow/Keras)
**Coût d'adoption : +1 callback dans model.fit()**

```python
# AVANT
model.fit(X, y, epochs=20)

# APRÈS (+1 argument)
model.fit(X, y, epochs=20, callbacks=[KabenaCallback(K=0.25)])
```

In [ ]:
import numpy as np
import tensorflow as tf
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

# Données simulées (embeddings NLP)
np.random.seed(42)
n, d = 1000, 50
X = np.random.randn(n, d).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)
X_train, X_test = X[:800], X[800:]
y_train, y_test = y[:800], y[800:]

# Modèle Keras
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(d,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2),
])
model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print('=== Entraînement AVEC K-ABENA (+1 callback) ===')
# ── AVANT : model.fit(X_train, y_train, epochs=30)
# ── APRÈS (+1 callback) :
history = model.fit(
    X_train, y_train,
    epochs=30, batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[KabenaCallback(K=0.25, N=0.3, verbose=True)],  # ← +1 ligne
    verbose=0,
)
print(f"\nAccuracy test : {model.evaluate(X_test, y_test, verbose=0)[1]:.4f}")

## Avec GradientTape (contrôle total)

In [ ]:
model2 = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(d,)),
    tf.keras.layers.Dense(2),
])
cfg     = KabenaConfig(K=0.25, N=0.3, task='classification', verbose=True)
trainer = KabenaTFTrainer(
    model2, cfg,
    optimizer=tf.keras.optimizers.SGD(0.05)
)
history = trainer.fit(
    tf.constant(X_train), tf.constant(y_train),
    epochs=30, batch_size=64
)